# Lesson 22 · Pretraining & fine-tuning: learn first from unlabeled data, then adjust a little for the task at hand

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/22-tien-huan-luyen-en.ipynb)

This notebook accompanies [Lesson 22 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/tien-huan-luyen/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/22-tien-huan-luyen.ipynb)):
the code, the data and every check are the same. The interactive demo is on the lesson page.
We reuse the course's MNIST MLP (784 → 256 → 128, exactly the encoder of Lesson 19) and ask three questions:
**how much does unlabeled pretraining help, and when?**, **what does a linear probe measure?** and
**what does LoRA lose, and what does it keep, by training fewer parameters?** At the end, we read the measurements
of a real CLIP model.

**What you'll do:**
1. Create labels from the data itself (the next token, a masked token), write SimCLR's NT-Xent loss yourself
   (checked with finite differences), run a small SimCLR on unlabeled images, then measure its encoder with a linear
   probe.
2. Read the results of the sweep over the number of labels (10–55,000, 3 seeds) and rerun the baselines: a random
   network, raw pixels.
3. Read the zero-shot and linear-probe measurements of CLIP ViT-B/32 on real photos and on MNIST.
4. Write LoRA $h = xW + \tfrac{\alpha}{r}\,xAB$, transfer a network from digits 0–4 to 5–9 with a probe, LoRA and
   full fine-tuning, then measure how much of the old task is forgotten.
5. Measure the rank of the update $\Delta W$, rebuild the demo's network with its $k$ largest directions, and compute
   the number of LoRA parameters of GPT-3.

**Background:** [Lesson 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/transformer/),
[Lesson 18 · Vision Transformer (ViT)](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/vision-transformer/) (CLIP's
image tower is a ViT), [Lesson 19 · Autoencoders & VAEs](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/autoencoder-vae/)
and the Adam optimizer of [Lesson 07](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/bo-toi-uu/). **Run time:**
under a minute on a CPU, including the first download of MNIST (~11 MB) and of the demo's 1.1 MB of results.

Every number in the notebook is computed by the cells below, or read from the very files the demo uses, and matches
the numbers in the demo. The full runs (9 methods × 5 label counts × 3 seeds, a learning-rate grid for each
combination, and CLIP run on 12,500 MNIST images) take tens of minutes on a many-core machine, so the notebook loads
their results and reruns a small version of each mechanism. CLIP has 151 million parameters; the notebook doesn't
download its weights, it only reads the measured cosines.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `fmt()` prints numbers in the
English format: a decimal point, and commas between thousands. That is the same point Python uses when you *type* a
decimal (`0.1`); printing only adds the thousands separators and a real minus sign. `span()` prints the min–max
range across 3 seeds, as the demo does.

In [ ]:
import base64
import gzip
import hashlib
import io
import itertools
import json
import math
import urllib.request
from decimal import ROUND_HALF_UP, Decimal
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# The course's palette (colorblind-safe), always paired with a second encoding (marker shape, dash style, label)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("orange_gray_blue", [ORANGE, "#f2f2f2", BLUE])  # negative → 0 → positive
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def fmt(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38 (never “−0”); exact halves round away from zero
    (95.25 → 95.3), as on the lesson's web page."""
    r = float(Decimal(float(x)).quantize(Decimal(1).scaleb(-int(digits)), ROUND_HALF_UP))
    s = f"{r + 0.0:,.{digits}f}"
    return s.replace("-", "−")


def pct(p, digits=2):
    return fmt(100 * p, digits) + "%"


def span(values, digits=2):
    """The min–max range (fractions → percent) across seeds: 80.63–82.04%."""
    lo, hi = fmt(100 * min(values), digits), fmt(100 * max(values), digits)
    return f"{lo}%" if lo == hi else f"{lo}–{hi}%"


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 60,000 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return fmt(v, len(s.split(".")[1]) if "." in s else 0)


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def log_tick(v, _pos=None):
    """A log-axis tick label: 0.5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def fmt_axes(fig):
    """Put the labels of every axis of the figure (colorbars included) in the English format; call right before
    plt.show(). A log axis gets 1–2–5 ticks if it is short, or powers of 10 (evenly spaced, at most ~9 ticks) if it
    is long."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


def sci(x, digits=1):
    """A very small number: 1.9·10⁻⁷."""
    e = math.floor(math.log10(abs(x)))
    return f"{fmt(x / 10**e, digits)}·10{str(e).translate(SUP)}"


def show_digits(axes, images, titles=None, fontsize=9):
    """Draw each 784-pixel image (values 0–1) in its own panel: white background, black strokes."""
    for i, ax in enumerate(np.ravel(axes)):
        ax.axis("off")
        if i < len(images):
            ax.imshow(np.asarray(images[i]).reshape(28, 28), cmap="gray_r", vmin=0, vmax=1)
            if titles is not None:
                ax.set_title(titles[i], fontsize=fontsize)


print(fmt(13007, 0), fmt(0.1, 1), fmt(-4.3805, 4), pct(0.993527), span([0.8063, 0.8204]), sci(1.9e-7))

## 2 · The data

MNIST (LeCun et al., 1998): 60,000 training images and 10,000 test images, 28×28 grayscale handwritten digits. The
cell below downloads the 4 original files and checks their MD5 hashes, so everyone who runs it gets exactly the same
data. As in the demo's runs, we split them into:
- **the training pool:** the first 55,000 images; SimCLR learns on the whole pool *without looking at the labels*,
  and the labeled sets are drawn from it;
- **validation:** the last 5,000 images of the training file (the demo chooses the learning rate $\eta$ here, never
  on the test set);
- **test:** the 10,000 images of the test file, used only to report results.

The cell also loads the four results files that the demo uses (`fetch_json`, with an MD5 check):
- `pretrain-label.json`: 9 methods × 5 label counts × 3 seeds (the demo's card 1);
- `pretrain-transfer.json`: transfer from 0–4 to 5–9 with a probe, LoRA of rank 1–16, full fine-tuning or training
  from scratch;
- `pretrain-transfer-weights.json`: seed 0's fully fine-tuned network, which the demo rebuilds in the browser;
- `pretrain-clip.json`: the measurements of CLIP ViT-B/32.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # MNIST's standard MD5 hashes
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"


def load(kind):
    """IDX format: the first 16 bytes of an image file (8 bytes of a label file) are the header; the rest are bytes
    0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 784)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


# The results of the long training runs (the same files as the demo on the web). When publishing, build.py replaces
# the placeholder string in the urlretrieve below with the data/ folder of the public notebook repo.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
LAB = fetch_json("pretrain-label.json", "7ff752f72033d5a8d2605c0ac5db1b80")
TRA = fetch_json("pretrain-transfer.json", "c2c712a895375ef0161d20797359b330")
SHIP = fetch_json("pretrain-transfer-weights.json", "906cf393e1eae9413d3f41563da66b6e")
CLIP = fetch_json("pretrain-clip.json", "c0f381b32aed08626b4b4b8c17090f4f")
size = sum(f.stat().st_size for f in Path("data").glob("pretrain-*.json"))
print(
    f"training: {fmt(len(train_img), 0)} images · test: {fmt(len(test_img), 0)} images · the demo's 4 files: {fmt(size / 1e6, 1)} MB"
)

Images become row vectors of 784 numbers in $[0, 1]$, as float32 like the demo's runs. `subset()` draws a balanced
labeled set (the same number of images for each class) from the pool of 55,000 images, exactly the way the demo does:
one fixed draw for each seed. For “55,000 labels”, each class takes at most 5,500 images, so a class that has fewer
gives all of them; the total is actually a little under 55,000.

In [ ]:
F32 = np.float32
X, y = train_img.astype(F32) / 255, train_lab.astype(np.int64)
Xt, yt = test_img.astype(F32) / 255, test_lab.astype(np.int64)
POOL, VAL = np.arange(55000), np.arange(55000, 60000)  # training pool · validation


def subset(per_class, seed, classes=range(10)):
    """per_class images of each class from the pool, one fixed draw for each seed (all of them if the class has
    fewer)."""
    rng = np.random.default_rng(40_000 + seed)
    out = []
    for c in classes:
        idx = POOL[y[POOL] == c]
        out.append(rng.choice(idx, per_class, replace=False) if per_class < len(idx) else idx)
    return np.sort(np.concatenate(out))


print(f"pool {fmt(len(POOL), 0)} images · validation {fmt(len(VAL), 0)} · test {fmt(len(yt), 0)}")
print("images per class in the pool:", np.bincount(y[POOL]).tolist(), "· each class takes at most", fmt(5500, 0))
print(
    f"the “100 labels” set: {len(subset(10, 0))} images · the “55,000 labels” set: {fmt(len(subset(5500, 0)), 0)} images"
)

## 3 · Labels already in the data

Labels assigned by humans are expensive and scarce; raw data is nearly limitless. **Self-supervised** learning
creates labels from the data itself, then learns with the familiar cross-entropy of Lesson 04:
- **GPT** predicts the next token: each position $t$ of a sentence is one example, with context $x_{<t}$ and label
  $x_t$, and the loss $\mathcal L_{\text{LM}} = -\sum_t \log p_\theta(x_t \mid x_{<t})$ (Radford et al., 2018).
- **BERT** masks 15% of the tokens, then predicts them back. Of the tokens chosen, 80% are replaced with `[MASK]`,
  10% with a random token and 10% are left unchanged: at fine-tuning time there is no `[MASK]` at all, so the model
  must not be allowed to pay attention only to it (Devlin et al., 2018).

The cell below creates both kinds of labels from one sentence, without anyone labeling anything. Here the sentence
is split at the spaces, so a token is one syllable or one punctuation mark (Lesson 14 uses BPE). The sentence is Vietnamese, the course's original language, and says: “labels assigned by
humans are expensive and scarce, while raw data is nearly limitless. self-supervised pretraining creates labels from
the data itself: predict the next token, or predict back the tokens that are masked”. Vietnamese writes each
syllable as a separate word, and the sentence also puts a space around each punctuation mark, so splitting at the
spaces gives the tokens.

In [ ]:
text = (  # the Vietnamese sentence translated above
    "nhãn do con người gán thì đắt và ít , còn dữ liệu thô thì gần như vô tận . tiền huấn luyện tự giám sát tự tạo "
    "nhãn từ chính dữ liệu : đoán token tiếp theo , hoặc đoán lại những token bị che"
)
tokens = text.split()
for t in range(1, 5):  # GPT: context x_<t → label x_t
    print(f"{' '.join(tokens[:t]):>18}  →  {tokens[t]}")
print(f"… a sentence of {len(tokens)} tokens gives {len(tokens) - 1} (context, label) pairs\n")


SHARE, P_MASK, P_RAND = 0.15, 0.8, 0.1  # BERT's recipe (Devlin et al., 2018)


def bert_mask(tokens, rng, share=SHARE):
    """Choose 15% of the positions; at each one: 80% → [MASK], 10% → a random token, 10% unchanged. The label is the
    original token."""
    pos = np.sort(rng.choice(len(tokens), max(1, round(share * len(tokens))), replace=False))
    out, vocab = list(tokens), sorted(set(tokens))
    for i in pos:
        u = rng.random()
        out[i] = "[MASK]" if u < P_MASK else vocab[rng.integers(len(vocab))] if u < P_MASK + P_RAND else tokens[i]
    return out, pos


print(f"BERT chooses {pct(SHARE, 0)} of the tokens; of those, {pct(P_MASK, 0)} → [MASK], {pct(P_RAND, 0)} → a random "
      f"token, {pct(1 - P_MASK - P_RAND, 0)} unchanged")  # fmt: skip
masked, pos = bert_mask(tokens, np.random.default_rng(1))
print(" ".join(masked))
print("labels to predict:", {int(i): tokens[i] for i in pos})
# with very many tokens, the 80/10/10 split shows
big = ["a", "b", "c", "d"] * 25_000
m_big, pos_big = bert_mask(big, np.random.default_rng(0))
kind = np.array(["mask" if m_big[i] == "[MASK]" else "kept" if m_big[i] == big[i] else "changed" for i in pos_big])
print(f"expected with 4 tokens: kept {pct(1 - P_MASK - P_RAND + P_RAND / 4, 1)}, changed {pct(P_RAND * 3 / 4, 1)}")
print(
    f"{fmt(len(pos_big), 0)} positions chosen out of {fmt(len(big), 0)} tokens:",
    {k: pct((kind == k).mean(), 1) for k in ("mask", "changed", "kept")},
)

At 100,000 tokens, the “kept” share is about 12.5%, not 10%: the vocabulary here has only 4 tokens, so the “random
token” lands on the original token 1 time in 4, and a quarter of the 10% that are “changed” end up “kept”.

Images have no “next token”, but there is another way to create labels: two transformed versions of the same image
should have the same representation. That is SimCLR (section 5). Before that, we build the lesson's network.

## 4 · The lesson's network, and three ways to train it

The network is the course's MNIST MLP: an encoder 784 → 256 → 128 (ReLU, He initialization as in Lesson 09; exactly
the encoder of Lesson 19) and a linear output layer, in the row convention:

$$h_1 = \mathrm{ReLU}(xW_1 + b_1), \qquad h_2 = \mathrm{ReLU}(h_1W_2 + b_2), \qquad z = h_2W_h + b_h$$

The three ways of training use this same network, and differ only in **which parameters are trained**:
- **linear probe**: freeze the encoder, train only $W_h, b_h$;
- **full fine-tuning**: train every parameter, starting from the pretrained encoder;
- **training from scratch**: like full fine-tuning, but the encoder starts from a random initialization.

Softmax regression on raw pixels is the case with no encoder: $z = xW_h + b_h$. The functions below copy exactly how
the demo initializes (each part has its own seed: the encoder `seed`, the output layer `10_000 + seed`), so the same
seed gives the same network.

In [ ]:
SIZES = [784, 256, 128]


def init_encoder(seed):
    """The 784 → 256 → 128 encoder, He initialization: W ~ N(0, 2/d_in), b = 0 (float32, as in the demo)."""
    rng, p = np.random.default_rng(seed), {}
    for i, (a, b) in enumerate(itertools.pairwise(SIZES)):
        p[f"W{i}"] = (rng.standard_normal((a, b)) * np.sqrt(2 / a)).astype(F32)
        p[f"b{i}"] = np.zeros(b, F32)
    return p


def add_head(p, k, seed):
    """A new output layer with k classes: N(0, 1/128) on top of the encoder; all zeros for softmax regression on the
    pixels."""
    rng, q = np.random.default_rng(10_000 + seed), dict(p)
    q["Wh"] = (rng.standard_normal((128, k)) * np.sqrt(1 / 128)).astype(F32) if p else np.zeros((784, k), F32)
    q["bh"] = np.zeros(k, F32)
    return q


def depth(p):
    return sum(1 for k in p if k[0] == "W" and k[1:].isdigit())


def softmax(z):
    e = np.exp(z - z.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def forward(p, x):
    """The hidden layers [x, h₁, h₂] and the logits z. If p also has Aᵢ, Bᵢ (LoRA, section 10), layer i adds (hAᵢ)Bᵢ."""
    hs = [x]
    for i in range(depth(p)):
        a = hs[-1] @ p[f"W{i}"] + p[f"b{i}"]
        if f"A{i}" in p:
            a = a + (hs[-1] @ p[f"A{i}"]) @ p[f"B{i}"]
        hs.append(np.maximum(a, 0))
    return hs, hs[-1] @ p["Wh"] + p["bh"]


def features(p, X, bs=5000):
    """The (frozen) encoder's output h₂ for the whole set; the pixels themselves if there is no encoder."""
    if depth(p) == 0:
        return X
    out = []
    for i in range(0, len(X), bs):
        h = X[i : i + bs]
        for j in range(depth(p)):
            h = np.maximum(h @ p[f"W{j}"] + p[f"b{j}"], 0)
        out.append(h)
    return np.concatenate(out)


def acc(p, X, y, off=0, bs=5000):
    """The fraction predicted correctly; off = 5 when a 5-class output layer reads the digits 5–9 (section 10)."""
    pred = np.concatenate([forward(p, X[i : i + bs])[1].argmax(1) for i in range(0, len(X), bs)])
    return float((pred + off == y).mean())

Training uses exactly the demo's recipe: Adam (Lesson 07), a cosine schedule down to 0, batches of 64 images in an
order that is reshuffled after each epoch, the cross-entropy of Lesson 04 and the backpropagation of Lesson 06.
Parameters that are not in `train` stay fixed. A linear probe computes the features $h_2$ of every image once in
advance, then trains only a softmax regression on them.

In [ ]:
class Adam:
    """Adam (Lesson 07) for the parameters named in keys; updates p in place."""

    def __init__(self, p, keys, b1=0.9, b2=0.999, eps=1e-8):
        self.m = {k: np.zeros_like(p[k]) for k in keys}
        self.v = {k: np.zeros_like(p[k]) for k in keys}
        self.t, self.b1, self.b2, self.eps = 0, b1, b2, eps

    def step(self, p, g, lr):
        self.t += 1
        c1, c2 = 1 - self.b1**self.t, 1 - self.b2**self.t
        for k in self.m:
            self.m[k] = self.b1 * self.m[k] + (1 - self.b1) * g[k]
            self.v[k] = self.b2 * self.v[k] + (1 - self.b2) * g[k] * g[k]
            p[k] = (p[k] - lr * (self.m[k] / c1) / (np.sqrt(self.v[k] / c2) + self.eps)).astype(p[k].dtype)


def lr_at(t, lr, steps, warm=0):
    """Linear warmup over the first warm steps (if any), then a cosine schedule down to 0."""
    if t < warm:
        return lr * (t + 1) / warm
    return lr * 0.5 * (1 + np.cos(np.pi * (t - warm) / max(1, steps - warm)))


def grads(p, x, y, train):
    """Cross-entropy and its gradient (backpropagation, Lesson 06); the hidden layers are computed only when train has
    their parameters."""
    hs, z = forward(p, x)
    n = len(y)
    dz = softmax(z)
    loss = float(-np.log(dz[np.arange(n), y] + 1e-12).mean())
    dz[np.arange(n), y] -= 1
    dz /= n
    g = {"Wh": hs[-1].T @ dz, "bh": dz.sum(0)}
    if set(train) - {"Wh", "bh"}:
        dh = dz @ p["Wh"].T
        for i in reversed(range(depth(p))):
            da = dh * (hs[i + 1] > 0)  # ReLU lets the gradient through where it is positive
            if f"W{i}" in train:
                g[f"W{i}"], g[f"b{i}"] = hs[i].T @ da, da.sum(0)
            if f"A{i}" in p:  # LoRA (section 10): a = hW + b + (hA)B
                g[f"B{i}"] = (hs[i] @ p[f"A{i}"]).T @ da
                dt = da @ p[f"B{i}"].T
                g[f"A{i}"] = hs[i].T @ dt
            if i > 0:
                dh = da @ p[f"W{i}"].T + (dt @ p[f"A{i}"].T if f"A{i}" in p else 0)
    return loss, g


def fit(p, X, y, train, lr, steps, seed):
    """Adam + cosine on (X, y), batches of 64, as in the demo. Linear probe: trains on the precomputed features."""
    rng = np.random.default_rng(30_000 + seed)
    probe = set(train) <= {"Wh", "bh"}
    q = {"Wh": p["Wh"], "bh": p["bh"]} if probe else p
    if probe:
        X = features(p, X)
    opt, n = Adam(q, train), len(X)
    bsz = min(64, n)
    order, pos = rng.permutation(n), 0
    for t in range(steps):
        if pos + bsz > n:  # end of an epoch: reshuffle
            order, pos = rng.permutation(n), 0
        idx = order[pos : pos + bsz]
        pos += bsz
        opt.step(q, grads(q, X[idx], y[idx], train)[1], lr_at(t, lr, steps))
    if probe:
        p["Wh"], p["bh"] = q["Wh"], q["bh"]
    return p


def steps_for(n):
    """As in the demo: at least 2,000 steps and at least 5 epochs over the labeled set."""
    return max(2000, math.ceil(5 * n / 64))


print(
    "steps for 100 labels:", steps_for(100), "· for the “55,000 labels” set:", fmt(steps_for(len(subset(5500, 0))), 0)
)

## 5 · SimCLR: two views of the same image

SimCLR (Chen et al., 2020) creates labels with random transformations. Take a batch of $B$ images and transform each
image twice, into $2B$ **views**. The two views of the same image are a **positive pair**; every other pair in the
batch is a negative pair. The network must recognize the positive partner of each view among $2B - 1$ candidates. It
doesn't know which digit any image is.

The transformation decides what the network learns. For digits, the demo uses a random affine transformation:
rotation by ±15°, scaling by 0.85–1.15 times and a shift of ±3 pixels, with bilinear interpolation and a background
of 0. The cell below writes it in numpy (the demo's `augment` uses `scipy.ndimage.map_coordinates`, with the same
result) and draws 6 views of 4 digits.

In [ ]:
def augment(X, rng, rot=15.0, scale=(0.85, 1.15), shift=3.0):
    """One random affine view of each image (B, 784): for each target pixel, find its source point, then
    interpolate."""
    B = len(X)
    img = X.reshape(B, 28, 28)
    th = np.deg2rad(rng.uniform(-rot, rot, B))
    sc = rng.uniform(scale[0], scale[1], B)
    sh = rng.uniform(-shift, shift, (B, 2))
    yy, xx = np.meshgrid(np.arange(28) - 13.5, np.arange(28) - 13.5, indexing="ij")  # coordinates around the center
    c, s = (np.cos(th) / sc)[:, None, None], (np.sin(th) / sc)[:, None, None]
    sy = c * yy - s * xx + 13.5 - sh[:, 0, None, None]  # the source point (row, column) of each target pixel
    sx = s * yy + c * xx + 13.5 - sh[:, 1, None, None]
    inside = (sy >= 0) & (sy <= 27) & (sx >= 0) & (sx <= 27)
    y0, x0 = np.clip(np.floor(sy), 0, 26).astype(int), np.clip(np.floor(sx), 0, 26).astype(int)
    fy, fx, b = sy - y0, sx - x0, np.arange(B)[:, None, None]
    v = (img[b, y0, x0] * (1 - fy) * (1 - fx) + img[b, y0 + 1, x0] * fy * (1 - fx)
         + img[b, y0, x0 + 1] * (1 - fy) * fx + img[b, y0 + 1, x0 + 1] * fy * fx)  # fmt: skip
    return np.where(inside, v, 0).reshape(B, 784).astype(X.dtype)


SIM = LAB["setup"]["simclr"]
assert (SIM["rot"], SIM["scale"], SIM["shift"]) == (15.0, [0.85, 1.15], 3.0)  # exactly the demo's transformation
print(
    f"rotate ±{fmt(SIM['rot'], 0)}°, scale {fmt(SIM['scale'][0])}–{fmt(SIM['scale'][1])} times, shift ±{fmt(SIM['shift'], 0)} pixels"
)
views_rng = np.random.default_rng(3)
fig, axes = plt.subplots(4, 7, figsize=(7.5, 4.6))
for row, i in zip(axes, [0, 1, 2, 7]):
    show_digits(row[:1], X[i : i + 1], ["original"])
    show_digits(row[1:], augment(np.repeat(X[i : i + 1], 6, 0), views_rng), ["view"] * 6)
plt.tight_layout()
plt.show()

The first column is the original image, and the 6 columns after it are random views of it: tilted, larger or
smaller, shifted in different ways, but still the same digit. If the network's representation is the same for every
view of an image and different between images, the network has had to learn what does not change when an image is
rotated, scaled or shifted.

### The NT-Xent loss

Each view goes through the encoder, then a **projection head** 128 → 128 → 64, which gives a vector $z$. Let
$s_{ij} = \cos(z_i, z_j)$ be the cosine between two views. For view $i$, whose positive partner is $p(i)$, the loss is
the cross-entropy of the problem “pick the right positive partner”, with temperature $\tau$ (demo: 0.5):

$$\ell_i = -\log \frac{\exp(s_{i,p(i)}/\tau)}{\sum_{k \ne i} \exp(s_{ik}/\tau)}, \qquad
\mathcal L = \frac{1}{2B}\sum_{i=1}^{2B} \ell_i$$

We write it first with a loop for a single view, then for the whole batch at once. Views $i$ and $i + B$ are a pair.

In [ ]:
TAU = 0.5


def ntxent_loop(z, i, tau=TAU):
    """ℓᵢ for a single view i, with a loop over every view k ≠ i."""
    B = len(z) // 2
    u = z / np.linalg.norm(z, axis=1, keepdims=True)
    pos = i + B if i < B else i - B
    denom = sum(np.exp(u[i] @ u[k] / tau) for k in range(len(z)) if k != i)
    return -np.log(np.exp(u[i] @ u[pos] / tau) / denom)


def ntxent(z, tau=TAU):
    """NT-Xent on 2B views: the loss, ∂L/∂z, and the fraction of views whose nearest neighbor is their positive
    partner."""
    n = len(z)
    nz = np.linalg.norm(z, axis=1, keepdims=True) + 1e-12
    u = z / nz
    S = (u @ u.T) / tau
    np.fill_diagonal(S, -np.inf)  # k ≠ i
    pos = np.concatenate([np.arange(n // 2, n), np.arange(n // 2)])
    m = S.max(1, keepdims=True)
    e = np.exp(S - m)
    loss = float((np.log(e.sum(1)) + m[:, 0] - S[np.arange(n), pos]).mean())
    dS = e / e.sum(1, keepdims=True)  # softmax by row, minus the one-hot of the positive partner (Lesson 04)
    dS[np.arange(n), pos] -= 1
    dS /= n
    dU = (dS + dS.T) @ u / tau  # S = UUᵀ/τ is symmetric: each sᵢⱼ contributes to both row i and row j
    dz = (dU - u * (u * dU).sum(1, keepdims=True)) / nz  # through the normalization u = z/‖z‖
    return loss, dz, float((S.argmax(1) == pos).mean())


z_toy = np.random.default_rng(0).standard_normal((8, 5))  # B = 4 images → 8 views, 5-dimensional z
loss_loop = np.mean([ntxent_loop(z_toy, i) for i in range(8)])
loss_vec, dz_toy, _ = ntxent(z_toy)
print(f"loop: {fmt(loss_loop, 6)} · whole batch: {fmt(loss_vec, 6)} · random guessing: log 7 = {fmt(np.log(7), 6)}")
assert np.isclose(loss_loop, loss_vec)

The two ways give the same number. If every $z$ were identical, each view would see 7 identical candidates and
$\ell_i = \log 7$. With random $z$, positive pairs are no closer than negative pairs, so the loss is about that size;
here it is even higher, because some negative pairs happen to be closer than the positive pairs. Learning means
pulling the loss below that mark.

The hand-written gradient in `ntxent` goes through three steps: the softmax by row (as in Lesson 04), the symmetric
cosine matrix $S = UU^\top/\tau$, and the normalization $u = z/\lVert z\rVert$. We check it with central finite
differences on every entry of $z$:

In [ ]:
H = 1e-6
num = np.zeros_like(z_toy)
for idx in np.ndindex(*z_toy.shape):
    zp, zm = z_toy.copy(), z_toy.copy()
    zp[idx] += H
    zm[idx] -= H
    num[idx] = (ntxent(zp)[0] - ntxent(zm)[0]) / (2 * H)
rel = np.abs(num - dz_toy).max() / np.abs(num).max()
print(f"largest relative error over {z_toy.size} entries: {sci(rel)}")
assert rel < 1e-6

## 6 · A small SimCLR, then a linear probe

SimCLR's encoder is exactly the 784 → 256 → 128 encoder of section 4; the projection head is used only during
pretraining, then thrown away. Backpropagation carries the gradient $\partial\mathcal L/\partial z$ from `ntxent`
through the projection head and the encoder.

The demo pretrains for 5,000 steps of 256 images each (512 views). Here we run 200 steps of 128 images, with the same
initialization (seed 0) and the same Adam with $\eta = 10^{-3}$, a 2% warmup, then cosine. No label is used. To see
what the network has learned, we measure the fraction of positive pairs found correctly on 256 validation images (512
views), before and after training.

In [ ]:
def sim_init(seed):
    """The encoder of section 4 (same seed) + a projection head 128 → ReLU 128 → 64."""
    p = init_encoder(seed)
    rng = np.random.default_rng(50_000 + seed)
    p["P0"] = (rng.standard_normal((128, 128)) * np.sqrt(2 / 128)).astype(F32)
    p["c0"] = np.zeros(128, F32)
    p["P1"] = (rng.standard_normal((128, 64)) * np.sqrt(1 / 128)).astype(F32)
    p["c1"] = np.zeros(64, F32)
    return p


def sim_forward(p, x):
    h1 = np.maximum(x @ p["W0"] + p["b0"], 0)
    h2 = np.maximum(h1 @ p["W1"] + p["b1"], 0)
    g1 = np.maximum(h2 @ p["P0"] + p["c0"], 0)
    return (x, h1, h2, g1), g1 @ p["P1"] + p["c1"]


def sim_grads(p, x):
    (x0, h1, h2, g1), z = sim_forward(p, x)
    loss, dz, hit = ntxent(z)
    g = {"P1": g1.T @ dz, "c1": dz.sum(0)}
    d = (dz @ p["P1"].T) * (g1 > 0)
    g["P0"], g["c0"] = h2.T @ d, d.sum(0)
    d = (d @ p["P0"].T) * (h2 > 0)
    g["W1"], g["b1"] = h1.T @ d, d.sum(0)
    d = (d @ p["W1"].T) * (h1 > 0)
    g["W0"], g["b0"] = x0.T @ d, d.sum(0)
    return loss, g, hit


def simclr(steps, batch, lr=1e-3, seed=0):
    """SimCLR as in the demo: at each step, a batch of unlabeled images from the pool, two views of each image, Adam +
    2% warmup + cosine."""
    p = sim_init(seed)
    opt, rng = Adam(p, list(p)), np.random.default_rng(60_000 + seed)
    for t in range(steps):
        xb = X[rng.integers(0, len(POOL), batch)]
        _, g, _ = sim_grads(p, np.concatenate([augment(xb, rng), augment(xb, rng)]))
        opt.step(p, g, lr_at(t, lr, steps, warm=max(1, steps // 50)))
    return p


ev_rng = np.random.default_rng(1)
ev = np.concatenate([augment(X[VAL[:256]], ev_rng), augment(X[VAL[:256]], ev_rng)])  # 256 positive pairs
sim0 = sim_init(0)
sim = simclr(steps=200, batch=128)
for name, z in [
    ("raw pixels", ev),
    ("random network", sim_forward(sim0, ev)[1]),
    ("after 200 steps", sim_forward(sim, ev)[1]),
]:
    loss, _, hit = ntxent(z)
    print(
        f"{name:>15}: NT-Xent {fmt(loss, 3)} · positive pairs found {pct(hit, 1)} (random guessing: {pct(1 / 511, 1)})"
    )
demo_sim = SIM["seeds"]
print(
    f"demo, 5,000 steps × 256 images: NT-Xent {fmt(demo_sim[0]['loss'], 3)}, positive pairs found {span([s['posAcc'] for s in demo_sim], 1)} (3 seeds, on training batches)"
)
assert ntxent(sim_forward(sim, ev)[1])[2] > 2 * ntxent(sim_forward(sim0, ev)[1])[2]

Even the cosine between two raw-pixel images finds the positive partner a small fraction of the time, because a
digit shifted by a few pixels still overlaps itself. The random network does even worse than that. 200 steps of
SimCLR already raise the rate several times over; the demo's 5,000 steps take it to 95.2–95.6%.

A **linear probe** asks a different question: freeze the encoder, train only the output layer from 100 labeled
images (10 per class), and how much does it get right on the 10,000 test images? We compare three encoders: none
(softmax regression on raw pixels), a random network (initialization seed 0, nothing learned yet) and that same
network after 200 steps of SimCLR. For the first two baselines, we use the $\eta$ that the demo chose on the
validation set; for the small SimCLR, we choose $\eta$ on the validation set, as the demo does. No test image is used
for choosing.

In [ ]:
def probe(enc, per_class, seed=0, lrs=None, method=None):
    """A linear probe on the (frozen) encoder enc; η is chosen on validation. Returns (test, val, η)."""
    idx = subset(per_class, seed)
    n = str(10 * per_class) if per_class < 5500 else "55000"
    if lrs is None:  # the η that the demo chose for this exact combination
        lrs = [next(r["lr"] for r in LAB["methods"][method]["runs"][n] if r["seed"] == seed)]
    best = None
    for lr in lrs:
        p = fit(add_head({k: enc[k] for k in ("W0", "b0", "W1", "b1") if k in enc}, 10, seed), X[idx], y[idx],
                ["Wh", "bh"], lr, steps_for(len(idx)), seed)  # fmt: skip
        val = acc(p, X[VAL], y[VAL])
        if best is None or val > best[1]:
            best = (acc(p, Xt, yt), val, lr)
    return best


stored = {m: LAB["methods"][m]["runs"] for m in LAB["methods"]}
pr100 = {
    "raw pixels": probe({}, 10, method="pixel"),
    "random network": probe(init_encoder(0), 10, method="randprobe"),
    "SimCLR 200 steps": probe(sim, 10, lrs=[3e-3, 1e-2]),
}
for name, (te, va, lr) in pr100.items():
    print(f"{name:>16}, 100 labels: test {pct(te)} (validation {pct(va)}, η = {fmt(lr, 4)})")
for name, m in [("raw pixels", "pixel"), ("random network", "randprobe")]:
    assert abs(pr100[name][0] - stored[m]["100"][0]["test"]) < 0.005, "differs from the demo: check the data"
assert pr100["SimCLR 200 steps"][0] > pr100["random network"][0] + 0.08
print(
    f"200 steps × 128 images = {pct(200 * 128 / (SIM['steps'] * SIM['batch']), 0)} of the images the demo's SimCLR sees"
)

Same network, same initialization, same 100 labels: with only 200 extra steps of unlabeled training, the linear probe
is already more than 10 points above the random network. But it is only around the level of softmax regression on
raw pixels: 200 steps are only a small fraction of the demo's steps (printed above), and each step has only half as
many images. Section 7 reads the results of the full version.

## 7 · When does pretraining help?

The demo runs 9 methods on the same network, with 10 to 55,000 labels, 3 seeds for each combination (seed $s$ fixes
the labeled set, the output layer and the batch order, so every method sees exactly the same images under the same
seed). Each combination tries a grid of 6 learning rates (extended if the best $\eta$ is at the edge
of the grid) and chooses $\eta$ on the validation set. Here we read 5 of the methods:
- **from scratch** and **from scratch + augmentation**: SimCLR's very rotation/scaling/shift applied to the labeled
  images at every step;
- **SimCLR, probe** and **SimCLR, full fine-tuning**: the SimCLR encoder trained for 5,000 steps on 55,000 unlabeled
  images;
- **autoencoder, full fine-tuning**: the encoder of the deep autoencoder of Lesson 19, trained to reconstruct the
  pixels of the same images.

### 🤔 Predict first
With 100 labels, SimCLR beats training from scratch by a few points. With 55,000 labels (nearly the whole pool is
labeled), by how many points does fine-tuning from SimCLR beat training from scratch? And where does from scratch +
augmentation stand relative to SimCLR when there are only 100 labels?

In [ ]:
NS = LAB["setup"]["ns"]
NAME = {"scratch": "from scratch", "scratchaug": "from scratch + augmentation", "simprobe": "SimCLR probe",
        "simfull": "SimCLR full", "aefull": "autoencoder full", "pixel": "raw pixels",
        "randprobe": "random network, probe", "aeprobe": "autoencoder, probe", "simlora4": "SimCLR + LoRA r 4"}  # fmt: skip


def runs(m, n, key="test"):
    """The results of the 3 seeds (in seed order 0, 1, 2) for method m with n labels."""
    return [r[key] for r in LAB["methods"][m]["runs"][str(n)]]


def gain(m, n):
    """How many points above training from scratch, seed by seed (same labeled set)."""
    return [100 * (a - b) for a, b in zip(runs(m, n), runs("scratch", n))]


def pts(v, digits=1):
    lo, hi = fmt(min(v), digits), fmt(max(v), digits)
    return lo if lo == hi else f"{lo}–{hi}" if min(v) >= 0 else f"{lo} to {hi}"


print(
    f"trained parameters: full {fmt(LAB['methods']['simfull']['params'], 0)} · probe {fmt(LAB['methods']['simprobe']['params'], 0)}"
)
for n in NS:
    print(
        f"{fmt(n, 0):>6} labels: "
        + " · ".join(f"{NAME[m]} {span(runs(m, n), 1)}" for m in ("scratch", "simprobe", "simfull", "scratchaug"))
    )
print()
for n in NS:
    g = {m: gain(m, n) for m in ("simprobe", "simfull", "scratchaug", "aefull")}
    print(
        f"{fmt(n, 0):>6} labels, gain over from scratch (points): "
        + " · ".join(f"{NAME[m]} {pts(v)}" for m, v in g.items())
    )

At 55,000 labels, SimCLR with full fine-tuning beats training from scratch by −0.1 to 0.2 points, that is,
**within the seed-to-seed variation**: the advantage has disappeared. The largest advantage is where labels are few:
5.5–11.3 points at 10 labels and 4.0–7.7 points at 100 labels (probe), then 0.3–0.5 points at 10,000 labels (full).
He, Girshick & Dollár (2018) saw something similar when transferring from ImageNet to COCO, and so did ULMFiT (Howard
& Ruder, 2018), whose 100 labeled texts matched training from scratch on 100 times more.

More surprising is the “from scratch + augmentation” line: no pretraining at all, just SimCLR's own transformation
applied to the 100 labeled images, and the network reaches 87.0–90.4%, above every pretrained model. What SimCLR
teaches here is mostly “rotated slightly, or shifted by a few pixels, it is still the same digit”; when you know in
advance which invariance you need, putting it straight into the data is cheaper. Zoph et al. (2020) saw the same
phenomenon at large scale: stronger augmentation reduces, and even reverses, the benefit of ImageNet pretraining on
COCO. The autoencoder, for its part, doesn't help at any number of labels: reconstructing pixels doesn't teach what
classification needs.

The figure below plots the same numbers. Left: test accuracy against the number of labels (the line goes through the
median of the 3 seeds, the vertical bars are min–max). Right: how many points above training from scratch, seed by
seed; the gray line is 0. The horizontal axis is the number of labels on a log scale; the methods are shifted
sideways a little so that their bars don't overlap.

In [ ]:
STYLE = {"scratch": (BLUE, "o", "-"), "scratchaug": (GREEN, "^", "-"), "simprobe": (ORANGE, "s", "-"),
         "simfull": (ORANGE, "D", "--"), "aefull": (GREY, "v", ":")}  # fmt: skip
lx = np.log10(NS)
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.9))
OFFSET = (-0.06, -0.02, 0.02, 0.06)  # shift each method sideways so that the min–max bars don't overlap
for o, m in zip(OFFSET, ("scratch", "scratchaug", "simprobe", "simfull")):
    c, mk, ls = STYLE[m]
    v = np.array([runs(m, n) for n in NS]) * 100
    a.plot(lx + o, np.median(v, 1), color=c, marker=mk, ls=ls, label=NAME[m], mfc="white" if ls == "--" else c)
    a.vlines(lx + o, v.min(1), v.max(1), color=c, lw=2)
for o, m in zip(OFFSET, ("simprobe", "simfull", "scratchaug", "aefull")):
    c, mk, ls = STYLE[m]
    g = np.array([gain(m, n) for n in NS])
    b.plot(lx + o, np.median(g, 1), color=c, marker=mk, ls=ls, label=NAME[m], mfc="white" if ls == "--" else c)
    b.vlines(lx + o, g.min(1), g.max(1), color=c, lw=2)
b.axhline(0, color=GREY, lw=1)
for ax, yl in ((a, "test accuracy (%)"), (b, "gain over from scratch (points)")):
    ax.set_xticks(lx, [fmt(n, 0) for n in NS])
    ax.set(xlabel="number of labels (log scale)", ylabel=yl)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(frameon=False, fontsize=8)
a.set_title("Accuracy vs. number of labels", fontsize=11)
b.set_title("Advantage over training from scratch", fontsize=11)
plt.tight_layout()
fmt_axes(fig)
plt.show()

Two more things in the table above. First, at 10 labels, training only the output layer on top of the SimCLR encoder
(45.8–50.4%) does better than full fine-tuning (38.4–44.3%): training 235,146 parameters from 10 images overfits
easily, and while the output layer is still random, its gradient distorts the learned features (Kumar et al., 2022,
analyze the second phenomenon, call it *feature distortion* and propose probing first, then fine-tuning). From 1,000
labels on, it is the other way around. Second, the SimCLR probe stops at 94.7–94.8% however many labels there are,
while every method that trains the whole network goes above 98%: a linear layer on frozen features has its ceiling.

## 8 · What does a linear probe measure?

The linear probe is often used as a measure of “representation quality”: the encoder that gives the higher probe is
“better”. Before trusting that number, set some baselines.

### 🤔 Predict first
An encoder that **has learned nothing** (random He initialization, seed 0), frozen, with only the output layer
trained on the “55,000 labels” set: what percentage of the test set does it get right? Random guessing is 10%. And
what about softmax regression on raw pixels?

In [ ]:
pr55 = {
    "random network": probe(init_encoder(0), 5500, method="randprobe"),
    "raw pixels": probe({}, 5500, method="pixel"),
}
for (name, (te, va, lr)), m in zip(pr55.items(), ("randprobe", "pixel")):
    print(
        f"{name:>15}, {fmt(len(subset(5500, 0)), 0)} labels: test {pct(te)} (η = {fmt(lr, 2)}) · demo, 3 seeds: {span(runs(m, 55000))}"
    )
    assert abs(te - stored[m]["55000"][0]["test"]) < 0.003, "differs from the demo: check the data"
for m in ("aeprobe", "simprobe", "aefull", "simfull"):
    print(f"{NAME[m]:>25}, 55,000 labels: {span(runs(m, 55000))}")

A random network already gives 86.07% (85.52–86.35% across the 3 seeds), and raw pixels 92.79%. The reason: 128
random ReLU features of an image still keep most of the image's information, and a linear layer trained on 53,964
images finds a way to separate them. A probe measures **whether the features can be separated by a hyperplane**,
with an output layer that has enough data to learn; it does not measure what the model “knows”.

The second table shows that a probe can even rank things the wrong way round. The autoencoder's encoder gives a
probe of 91.81–92.15%, lower even than raw pixels, and much lower than SimCLR (94.7–94.8%). But full fine-tuning
from it reaches 98.3–98.4%, on par with SimCLR (98.4–98.5%). Hewitt & Liang (2019) propose always comparing a probe
against a **control baseline** (a task that the representation cannot “know”, or a random network as here) and
reading only the difference.

## 9 · CLIP: matching images with captions

CLIP (Radford et al., 2021) learns from 400 million image–text pairs from the Internet. An image tower (here a
ViT-B/32, Lesson 18) and a text tower (a Transformer, Lesson 17) map images and captions into the same space. In each
batch of $N$ pairs, let $S_{ij}$ be the cosine between image $i$ and caption $j$; the loss pulls the $N$ correct
pairs up and pushes the $N^2 - N$ wrong pairs down:

$$\mathcal L_{\text{CLIP}} = \tfrac12\big(\mathrm{CE}(S/\tau) + \mathrm{CE}(S^\top/\tau)\big)$$

CE takes the correct pair $i = j$ as the label for each row: $S/\tau$ in the image → text direction, $S^\top/\tau$ in
the reverse direction. The temperature $\tau$ is learned, and capped so that $1/\tau$ does not exceed 100. This is the
NT-Xent of section 5, except that a positive pair is made of an image and a sentence.

The demo measures a real CLIP ViT-B/32 (OpenAI's weights, 151,277,313 parameters) on the 8 Commons photos of Lesson
18 and on MNIST. The notebook doesn't download the model; it reads the measured cosines from `pretrain-clip.json` and
does the rest itself.

In [ ]:
cos = {k: np.array(v) for k, v in CLIP["cos"].items()}  # cos[language][image i, caption j]
SCALE = CLIP["model"]["scale"]  # the 1/τ that CLIP learned


def ce_rows(M):
    """The mean cross-entropy when the label of row i is column i."""
    return float(-np.log(np.diag(softmax(M))).mean())


print(f"{CLIP['model']['name']}: {fmt(CLIP['model']['params'], 0)} parameters, 1/τ = {fmt(SCALE, 2)}")
for lang in ("en", "vi"):
    S = cos[lang]
    right = int((S.argmax(1) == np.arange(8)).sum())
    loss = 0.5 * (ce_rows(SCALE * S) + ce_rows(SCALE * S.T))
    print(
        f"{lang} captions: e.g. “{CLIP['captions'][lang][1]}” · images that pick the right caption {right}/8 · L_CLIP = {fmt(loss, 3)}"
    )
print(f"random guessing: L = log 8 = {fmt(np.log(8), 3)}")
st = CLIP["stats"]
print(
    f"image–caption cosine, correct pairs: {fmt(st['matched'][0], 2)}–{fmt(st['matched'][1], 2)} · wrong pairs, mean {fmt(st['unmatchedMean'], 2)}"
)
print(f"mean image–image cosine {fmt(st['imgImgMean'], 2)} · caption–caption {fmt(st['txtTxtMean'], 2)}")

With English captions, each image has its highest cosine with its own caption, and $1/\tau = 100$ turns that small
margin (0.28–0.33 against a mean of 0.18) into a probability close to 1, so the loss is close to 0. With Vietnamese
captions (“một bức ảnh con mèo.” means “a photo of a cat.”), only 5/8 images pick the right one: CLIP learned from
Internet text, mostly English, and its tokenizer cuts Vietnamese sentences into far more pieces. The image–image and
text–text cosines are both higher than the image–text cosines: images and text lie in two separate regions of the
same space (the modality gap, Liang et al., 2022), so the cosine of a “correct” image–caption pair is only about 0.3.

The figure below draws the 8 photos and the two cosine matrices (rows: images, columns: captions; the cells outlined
in black are the correct pairs, and the white dot in the corner of a cell marks each image's highest-cosine caption).
The two matrices share the row labels on the left.

In [ ]:
photos = CLIP["photos"]
fig, axes = plt.subplots(1, 8, figsize=(10, 1.7))
for ax, ph in zip(axes, photos):
    ax.imshow(plt.imread(io.BytesIO(base64.b64decode(ph["jpeg"])), format="jpeg"))
    ax.set_title(ph["noun"], fontsize=8)
    ax.axis("off")
plt.show()
print("photos from Wikimedia Commons:", "; ".join(f"{ph['artist']} ({ph['license']})" for ph in photos))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
for ax, lang, title in zip(axes, ("en", "vi"), ("English captions", "Vietnamese captions")):
    S = cos[lang]
    im = ax.imshow(S, cmap="Blues", vmin=0.1, vmax=0.34)
    for i in range(8):
        ax.add_patch(plt.Rectangle((i - 0.5, i - 0.5), 1, 1, fill=False, ec="black", lw=1.5))
        ax.plot(S[i].argmax() + 0.33, i - 0.3, "o", color="white", mec="black", ms=4)
        for j in range(8):
            ax.text(
                j,
                i,
                fmt(S[i, j], 2),
                ha="center",
                va="center",
                fontsize=6.5,
                color="white" if S[i, j] > 0.27 else "black",
            )
    ax.set_xticks(range(8), [ph["noun"] for ph in photos], rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(8), [ph["noun"] for ph in photos] if lang == "en" else [], fontsize=8)
    ax.set_title(f"image–caption cosine, {title}", fontsize=10)
fig.colorbar(im, ax=axes, fraction=0.025)
fmt_axes(fig)
plt.show()
print("tokens per caption (en / vi):", CLIP["ntok"]["en"], "/", CLIP["ntok"]["vi"])

### Zero-shot: picking the nearest caption from a list we write ourselves

To classify the 1,000 ImageNet classes without any further training, we write 1,000 captions from a **prompt
template** (“a photo of a {class}.”), take their text vectors, then pick the class with the highest cosine with the
image. The probabilities are the softmax of $100 \cdot \cos$ over the 1,000 classes. The demo stores the table of
cosines for 8 images × 1,000 classes (float16) for 5 templates and for the average of the 7 templates that OpenAI
chose. The cell below reads the top class for each image. The sunflower photo has no correct class: ImageNet-1k has
no “sunflower” class.

In [ ]:
classes = CLIP["classes"]
zs = {
    t["id"]: np.frombuffer(base64.b64decode(CLIP["zs"][t["id"]]), "<f2").astype(np.float64).reshape(8, 1000)
    for t in CLIP["templates"]
}
TEMPL = {
    t["id"]: t["text"][0] if len(t["text"]) == 1 else f"average of {len(t['text'])} templates"
    for t in CLIP["templates"]
}
for tid in ("bare", "photo", "ens7"):
    top = zs[tid].argmax(1)
    p = softmax(SCALE * zs[tid])
    ok = [int(top[i]) in ph["expect"] for i, ph in enumerate(photos) if ph["expect"]]
    print(
        f"{TEMPL[tid]!r:>24}: correct {sum(ok)}/{len(ok)} · "
        + ", ".join(f"{classes[top[i]]} {fmt(p[i, top[i]], 2)}" for i in range(8))
    )

The template changes the answer. With the bare class name alone (“{}”), the cat becomes a “ruffed grouse” (a game
bird); with “a photo of a” added, it is correctly a “tabby cat”. For the sunflower, CLIP is forced to pick one of the
1,000 classes it has, and picks “cardoon” (a kind of artichoke): zero-shot is a closed-set problem, it can only answer
with what we put in the list. Add “sunflower” as class 1,001:

In [ ]:
for tid in ("bare", "photo", "ens7"):
    s = np.append(zs[tid][7], CLIP["sunflower"][tid][7])
    print(f"{TEMPL[tid]!r:>24}: add the class “sunflower” → probability {pct(softmax(SCALE * s[None])[0, -1], 1)}")
print(f"(class number {fmt(len(classes) + 1, 0)})")

When the list has the correct class, CLIP picks it with a probability of 68.7–96.1%, depending on the template.
Averaging many templates is usually more stable than a single one; that is how OpenAI reports its ImageNet results.

### CLIP on handwritten digits

On ImageNet, CLIP ViT-B/32 reaches 63.2% zero-shot without seeing a single ImageNet label (Radford et al., 2021,
Table 11). The paper also reports 51.9% on MNIST, with the template `a photo of the number: "{}".`. The demo measures
it again on the 10,000 test images, with 6 templates × 3 ways of naming the digits (the digit “7”, the English word
“seven”, the Vietnamese word “bảy”). One of the templates is Vietnamese too: `chữ số {} viết tay.` (“handwritten
digit {}.”).

### 🤔 Predict first
How many labeled images **per class** does softmax regression on raw pixels (the simplest model of the course) need
to beat CLIP zero-shot on MNIST: 1, 2, 10, or 100?

In [ ]:
zsm = CLIP["mnist"]["zeroShot"]
for names in ("digit", "word", "vi"):
    row = [z for z in zsm if z["names"] == names]
    print(f"names {names:>5}: " + " · ".join(f"{z['template']!r} {pct(z['acc'], 1)}" for z in row))
paper = next(z for z in zsm if z["names"] == "digit" and z["template"] == 'a photo of the number: "{}".')
print(f"the paper's template: {pct(paper['acc'])} (paper: {fmt(CLIP['mnist']['paper']['zeroShot'], 1)}%; ImageNet: "
      f"{fmt(CLIP['mnist']['paper']['imagenetZeroShot'], 1)}%)")  # fmt: skip
M = CLIP["mnist"]
print(f"the demo ran {fmt(M['nTest'] + M['nPool'] + M['nVal'], 0)} MNIST images through CLIP: {fmt(M['nTest'], 0)} test, "
      f"{fmt(M['nPool'], 0)} for the probes, {fmt(M['nVal'], 0)} validation")  # fmt: skip
print("images CLIP predicts as 0…9:", paper["predCounts"])

# Softmax regression on raw pixels, trained on k images per class: the same pool of 2,000 images and the same η
# (chosen on 500 validation images)
rng = np.random.default_rng(123)
clip_pool = np.sort(np.concatenate([rng.choice(np.flatnonzero(y[:55000] == c), 200, replace=False) for c in range(10)]))
pixel_k = {}
for k in (1, 2):
    for seed in range(3):
        r_ = np.random.default_rng(900 + seed)
        idx = clip_pool[
            np.concatenate([r_.choice(np.flatnonzero(y[clip_pool] == c), k, replace=False) for c in range(10)])
        ]
        ref = next(r for r in CLIP["mnist"]["probe"]["pixel"] if r["k"] == k)["runs"][seed]
        p = fit(
            {"Wh": np.zeros((784, 10), F32), "bh": np.zeros(10, F32)},
            X[idx],
            y[idx],
            ["Wh", "bh"],
            ref["lr"],
            1000,
            seed,
        )
        pixel_k[k, seed] = acc(p, Xt, yt)
        assert abs(pixel_k[k, seed] - ref["test"]) < 0.003, "differs from the demo: check the data"
    print(
        f"raw pixels, {k} image{'s' if k > 1 else ''} per class ({10 * k} images): {span([pixel_k[k, s] for s in range(3)])} (3 seeds)"
    )
assert min(pixel_k[2, s] for s in range(3)) > paper["acc"]

Just **2 images per class**: softmax regression on 20 labeled images already beats CLIP zero-shot (48.25%) on all 3
seeds. The CLIP paper says so itself: even a logistic regression baseline on raw pixels beats zero-shot CLIP on MNIST,
and its training data has almost no images that look like handwritten digits. The template changes the result a lot
(25.1–48.3% with the digits as names), and some templates make CLIP predict almost every image as the same digit.
Naming the digits in Vietnamese is close to random guessing (6.6–11.1%).

CLIP's representation is still useful: a linear probe on its (frozen) image vectors, trained on a few images per
class, goes far beyond raw pixels. The figure below plots both lines against the number of images per class (3
seeds, the vertical bars are min–max); the dashed line is zero-shot.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.6))
for kind, c, mk, label in (
    ("clip", ORANGE, "s", "probe on CLIP's image vectors"),
    ("pixel", BLUE, "o", "softmax regression on pixels"),
):
    rows = CLIP["mnist"]["probe"][kind]
    ks = np.array([r["k"] for r in rows])
    v = np.array([[q["test"] for q in r["runs"]] for r in rows]) * 100
    ax.plot(ks, np.median(v, 1), color=c, marker=mk, label=label)
    ax.vlines(ks, v.min(1), v.max(1), color=c, lw=2)
ax.axhline(100 * paper["acc"], color=GREY, ls="--", lw=1.5, label="CLIP zero-shot (0 labeled images)")
ax.set_xscale("log")
ax.set(xlabel="labeled images per class (log scale)", ylabel="accuracy on 10,000 test images (%)", ylim=(30, 100))
ax.set_title("MNIST: CLIP zero-shot, a probe on CLIP, and raw pixels", fontsize=11)
ax.grid(axis="y", alpha=0.3)
ax.legend(frameon=False, fontsize=8, loc="lower right")
fmt_axes(fig)
plt.show()
for kind in ("clip", "pixel"):
    print(
        kind, " · ".join(f"k {r['k']}: {span([q['test'] for q in r['runs']], 1)}" for r in CLIP["mnist"]["probe"][kind])
    )

## 10 · LoRA: freeze $W$, learn a low-rank update

Full fine-tuning retrains every parameter. With GPT-3 175B, every new task is a copy of 175 billion parameters.
**LoRA** (Hu et al., 2021) keeps $W$ unchanged and learns only an update of small rank $r$, in the course's row
convention:

$$h = xW + \tfrac{\alpha}{r}\, x A B, \qquad A \in \mathbb R^{d_{\text{in}} \times r}, \;\; B \in \mathbb R^{r \times d_{\text{out}}}$$

- $A$ is initialized randomly from $\mathcal N(0, 1/d_{\text{in}})$, and $B = 0$: at the start $AB = 0$, and the
  network is **exactly** the old network.
- $\Delta W = \tfrac{\alpha}{r} AB$ has rank at most $r$. It needs $r(d_{\text{in}} + d_{\text{out}})$ parameters
  instead of $d_{\text{in}} \cdot d_{\text{out}}$.
- $\alpha$ is a scaling constant. The paper sets $\alpha$ to the first value of $r$ they try and does not tune it
  afterwards; the demo uses $\alpha = r$, so $\alpha/r = 1$.
- The paper writes in the column convention $h = W_0x + BAx$, so its $A$, $B$ are $A^\top$, $B^\top$ here: the
  matrix that touches $x$ first is still the random one, and the other matrix is 0.

`forward` and `grads` in section 4 already have the LoRA branch (when `p` has `A0`, `B0`…). The cell below adds LoRA
to both hidden layers, counts the parameters to train, checks that $B = 0$ gives exactly the old network, then checks
the gradients of $A$, $B$ with finite differences.

In [ ]:
def add_lora(p, r, seed):
    """Rank-r LoRA on both hidden layers: Aᵢ ~ N(0, 1/d_in), Bᵢ = 0 (as in the demo; α/r = 1)."""
    rng, q = np.random.default_rng(20_000 + seed), dict(p)
    for i in range(depth(p)):
        a, b = p[f"W{i}"].shape
        q[f"A{i}"] = (rng.standard_normal((a, r)) / np.sqrt(a)).astype(F32)
        q[f"B{i}"] = np.zeros((r, b), F32)
    return q


LORA_KEYS = ["Wh", "bh", "A0", "B0", "A1", "B1"]
FULL_N = 784 * 256 + 256 + 256 * 128 + 128 + 128 * 5 + 5  # every parameter, with a 5-class output layer
for r in (1, 2, 4, 8, 16):
    n = sum(r * (a + b) for a, b in itertools.pairwise(SIZES)) + 128 * 5 + 5  # two pairs A, B + the new output layer
    print(f"LoRA r = {r:>2}: {fmt(n, 0):>6} trainable parameters ({pct(n / FULL_N, 1)} of {fmt(FULL_N, 0)})")
    assert n == TRA["methods"][f"lora{r}"]["params"], "differs from the demo: check the data"
assert FULL_N == TRA["methods"]["full"]["params"] == 234_501

net = add_lora(add_head(init_encoder(2), 5, 2), 4, 2)
plain = {k: v for k, v in net.items() if k[0] not in "AB"}
print(
    "B = 0: the network with LoRA gives exactly the same logits as the old network:",
    np.array_equal(forward(net, Xt[:64])[1], forward(plain, Xt[:64])[1]),
)
# the paper's column convention, h = W₀x + B_p A_p x, gives the same result with A_p = Aᵀ (random) and B_p = Bᵀ
x1, W, A, B = Xt[0].astype(np.float64), net["W0"], net["A0"], np.random.default_rng(4).normal(0, 0.1, net["B0"].shape)
assert np.allclose(x1 @ W + (x1 @ A) @ B, W.T @ x1 + B.T @ (A.T @ x1))
# the gradients of A, B (B nonzero so that every branch runs), float64, central differences
g_rng = np.random.default_rng(3)
q64 = {k: v.astype(np.float64) for k, v in net.items()}
q64["B0"], q64["B1"] = g_rng.normal(0, 0.1, q64["B0"].shape), g_rng.normal(0, 0.1, q64["B1"].shape)
xb, yb = Xt[:16].astype(np.float64), yt[:16] % 5
_, g = grads(q64, xb, yb, LORA_KEYS)
worst = 0.0
for k in ("A0", "B0", "A1", "B1"):
    for j in [tuple(g_rng.integers(0, s) for s in q64[k].shape) for _ in range(6)]:
        old = q64[k][j]
        q64[k][j] = old + 1e-6
        lp = grads(q64, xb, yb, [])[0]
        q64[k][j] = old - 1e-6
        lm = grads(q64, xb, yb, [])[0]
        q64[k][j] = old
        num = (lp - lm) / 2e-6
        worst = max(worst, abs(num - g[k][j]) / max(1e-6, abs(num) + abs(g[k][j])))
print(f"gradients of A, B: largest relative error {sci(worst)} over 24 entries")
assert worst < 1e-5

Rank-4 LoRA trains 6,341 parameters instead of 234,501, about 37 times fewer. In this small network, the new output
layer (645 parameters) is a sizable share; in a large Transformer, the LoRA part dominates completely (section 13).

### Switching tasks: from digits 0–4 to 5–9

The demo's experiment: a network trained **with labels** to read the digits 0–4 (28,076 images of the pool) is then
adjusted to read 5–9 (26,924 images), with a new 5-class output layer. Then we measure two things:
- **the new task:** the accuracy on the 4,861 test images of 5–9;
- **the old task:** put the **old** output layer back on the adjusted encoder (with LoRA merged into $W$), then
  measure on the 5,139 test images of 0–4. The drop is what the network has **forgotten** (Kirkpatrick et al., 2017,
  call it *catastrophic forgetting*).

The cell below trains seed 0's base network on 0–4 as the demo does: Adam with $\eta = 10^{-3}$, 3,000 steps.

In [ ]:
tr_old, tr_new = POOL[y[POOL] < 5], POOL[y[POOL] >= 5]
te_old, te_new = np.flatnonzero(yt < 5), np.flatnonzero(yt >= 5)
BASE = TRA["setup"]["base"]  # η 10⁻³, 3,000 steps
base = fit(
    add_head(init_encoder(0), 5, 0), X[tr_old], y[tr_old], LORA_KEYS[:2] + ["W0", "b0", "W1", "b1"], BASE["lr"],
    BASE["steps"], 0,
)  # fmt: skip
base_acc = acc(base, Xt[te_old], yt[te_old])
print(
    f"{fmt(len(tr_old), 0)} images of 0–4 · {fmt(len(tr_new), 0)} images of 5–9 · test {fmt(len(te_old), 0)} / {fmt(len(te_new), 0)}"
)
print(f"base network: η = {fmt(BASE['lr'], 3)}, {fmt(BASE['steps'], 0)} steps")
print(f"base network on 0–4: {pct(base_acc)} (demo, 3 seeds: {span([b['test'] for b in TRA['base']])})")
assert abs(base_acc - TRA["base"][0]["test"]) < 0.003, "differs from the demo: check the data"

The results can differ from the demo by a few images: 2 BLAS threads (as on Colab) add up the float32 products in a
different order from the demo's 1 thread, and after thousands of Adam steps, a difference in the last digit turns
into a few images whose answer changes. That is why the `assert`s of this section have a tolerance.

### Three ways to switch tasks
We adjust the base network to 5–9 with a probe (training only the new output layer), rank-4 LoRA and full
fine-tuning, each with the $\eta$ that the demo chose on validation for seed 0. Before running, try to rank the three
methods by accuracy on the new task, then by how well they keep the old task. Does any method forget nothing at all?

In [ ]:
n_new = steps_for(len(tr_new))
chosen = {m: next(r for r in TRA["methods"][m]["runs"]["all"] if r["seed"] == 0) for m in ("probe", "lora4", "full")}


def encoder_of():
    return {k: base[k].copy() for k in ("W0", "b0", "W1", "b1")}  # a copy of the base network's encoder


nets = {
    "probe": (add_head(encoder_of(), 5, 100), ["Wh", "bh"]),
    "lora4": (add_lora(add_head(encoder_of(), 5, 100), 4, 0), LORA_KEYS),
    "full": (add_head(encoder_of(), 5, 100), ["Wh", "bh", "W0", "b0", "W1", "b1"]),
}  # the new output layer: seed 100, independent of the old output layer (as in the demo)


def merged(p):
    """The encoder with LoRA merged in: Wᵢ + AᵢBᵢ (α/r = 1)."""
    return {f"W{i}": p[f"W{i}"] + (p[f"A{i}"] @ p[f"B{i}"] if f"A{i}" in p else 0) for i in range(2)} | {
        "b0": p["b0"], "b1": p["b1"]}  # fmt: skip


res = {}
for m, (p, keys) in nets.items():
    fit(p, X[tr_new], y[tr_new] - 5, keys, chosen[m]["lr"], n_new, 0)
    new = acc(p, Xt[te_new], yt[te_new], off=5)
    old = acc(merged(p) | {"Wh": base["Wh"], "bh": base["bh"]}, Xt[te_old], yt[te_old])
    res[m] = (new, old)
    print(f"{m:>6} (η = {fmt(chosen[m]['lr'], 3)}): new task 5–9 {pct(new)} · old task 0–4 {pct(old)}  "
          f"(demo, seed 0: {pct(chosen[m]['new'])} / {pct(chosen[m]['old'])})")  # fmt: skip
    assert abs(new - chosen[m]["new"]) < 0.008 and abs(old - chosen[m]["old"]) < 0.02, "differs from the demo"
assert res["probe"][1] == base_acc and res["lora4"][1] > res["full"][1] and res["full"][0] > res["lora4"][0]
assert res["full"][0] - res["lora4"][0] < 0.01 and 0.01 < res["lora4"][1] - res["full"][1] < 0.02  # as said below

- **The probe** forgets nothing: the encoder stays fixed, so with the old output layer, the old task gives exactly
  the base network's number. But the features learned for 0–4 only take the new task to 94.69%.
- **Full fine-tuning** learns the new task best, and forgets the most.
- **Rank-4 LoRA** is in between: less than a point below full fine-tuning on the new task, and about one to two
  points better at keeping the old task.

One seed is not enough to draw a conclusion. The figure below plots every method of the demo at the “all images”
level, 3 seeds per method: the horizontal axis is the new task, the vertical axis the old task. The top right corner
is best.

In [ ]:
METHODS = [("probe", "probe", GREY, "o"), ("lora1", "LoRA r 1", GREEN, "v"), ("lora4", "LoRA r 4", GREEN, "s"),
           ("lora16", "LoRA r 16", GREEN, "D"), ("full", "full", ORANGE, "^")]  # fmt: skip
fig, ax = plt.subplots(figsize=(7, 4.2))
for m, label, c, mk in METHODS:
    rs = TRA["methods"][m]["runs"]["all"]
    ax.scatter([100 * r["new"] for r in rs], [100 * r["old"] for r in rs], color=c, marker=mk, s=60, label=label,
               edgecolor="black", lw=0.6, zorder=3)  # fmt: skip
ctrl = TRA["headControl"]["runs"]
ax.scatter([100 * r["new"] for r in ctrl], [100 * r["old"] for r in ctrl], color="white", edgecolor=ORANGE, marker="^",
           s=60, lw=1.5, label="full, new output layer = the old layer's initialization", zorder=3)  # fmt: skip
ax.set(xlabel="new task: accuracy on 5–9 (%)", ylabel="old task: accuracy on 0–4 (%)")
ax.set_title("Learning a new task, forgetting the old one (all 5–9 images, 3 seeds)", fontsize=11)
ax.grid(alpha=0.3)
ax.legend(frameon=False, fontsize=8, loc="lower left")
fmt_axes(fig)
plt.show()
for m in ("probe", "lora1", "lora2", "lora4", "lora8", "lora16", "full", "scratch"):
    rs = TRA["methods"][m]["runs"]["all"]
    old = f" · old task {span([r['old'] for r in rs], 1)}" if "old" in rs[0] else ""
    print(
        f"{m:>7} ({fmt(TRA['methods'][m]['params'], 0):>7} parameters): new task {span([r['new'] for r in rs], 1)}{old}"
    )
print(
    f"full, new output layer = the old layer's initialization: new task {span([r['new'] for r in ctrl], 1)} · old task {span([r['old'] for r in ctrl], 1)}"
)
gap = {m: [100 * (f["new"] - r["new"]) for f, r in zip(TRA["methods"]["full"]["runs"]["all"], TRA["methods"][m]["runs"]["all"])]
       for m in ("lora2", "lora4")}  # fmt: skip


def old_of(ms):
    """The old task (0–4) for every seed of the methods ms, at the “all images” level."""
    return [r["old"] for m in ms for r in TRA["methods"][m]["runs"]["all"]]


print(f"LoRA r 2–4 trails full on the new task (same seed): {pts(gap['lora2'] + gap['lora4'], 2)} points")
print(f"old task: LoRA r 2–4 {span(old_of(['lora2', 'lora4']))} · full {span(old_of(['full']))}"
      f" · worst seed of r 8 and of r 16: {span([min(old_of(['lora8'])), min(old_of(['lora16']))])}")  # fmt: skip

Across the 3 seeds, the order holds:
- **New task:** probe 94.5–95.0% < LoRA r 4 98.1–98.2% < full 98.7–98.8%. LoRA of rank 2–4 trails full fine-tuning
  by 0.55–1.38 points, and the gap narrows as the rank grows.
- **Old task:** LoRA of rank 2–4 keeps 95.25–98.62%, full fine-tuning only 80.17–94.82%. No method that adjusts the
  encoder gets away without forgetting anything; only the probe, which doesn't touch the encoder, forgets nothing.
- **Ranks 8–16:** they learn the new task nearly as well as full fine-tuning, but their worst seed forgets as much as
  full fine-tuning (the old task is down to 78.38–86.55%). How much is forgotten varies a lot between seeds, because
  $\eta$ is chosen for the new task, not to reduce forgetting.
- **A small detail changes how much is forgotten:** if the new output layer starts from exactly the initial values of
  the old output layer (the hollow triangles), full fine-tuning learns the new task as usual, but the old task keeps
  only 43.1–55.8%. The amount forgotten in one run is not a constant of the method.

Biderman et al. (2024) found the same picture in large language models: “LoRA learns less and forgets less”. Fewer
parameters means less forgetting, not no forgetting.

## 11 · Is the full fine-tuning update low-rank?

LoRA rests on a hypothesis: the update $\Delta W$ made when adapting to a new task has a low “intrinsic rank” (Hu et
al., 2021, inspired by Aghajanyan et al., 2020). We test it on the very full fine-tuning we just ran: a singular
value decomposition $\Delta W_1 = W_1^{\text{after}} - W_1^{\text{before}} = U\,\mathrm{diag}(\sigma)\,V^\top$, with
$\sigma_1 \ge \sigma_2 \ge \dots$, then the **effective rank**

$$\operatorname{erank}(\Delta W) = \exp\Big(-\sum_k p_k \log p_k\Big), \qquad p_k = \frac{\sigma_k^2}{\sum_j \sigma_j^2}$$

It equals $r$ if the update is spread evenly over $r$ directions, and 1 if there is only one direction. The demo uses
the energy shares $\sigma_k^2$; the original definition by Roy & Vetterli (2007) uses $p_k = \sigma_k / \sum_j
\sigma_j$ and gives larger numbers. We print both.

### 🤔 Predict first
The $\Delta W_1$ of full fine-tuning is 784 × 256, so its rank is at most 256. Is its effective rank close to 4 (like
rank-4 LoRA), about 50, or close to 256?

In [ ]:
def erank(dW, power=2):
    """The effective rank exp(H(p)) with p ∝ σᵏ (k = 2: energy shares, as in the demo; k = 1: Roy & Vetterli 2007)."""
    s = np.linalg.svd(np.asarray(dW, np.float64), compute_uv=False)
    p = s**power / (s**power).sum()
    p = p[p > 0]
    return float(np.exp(-(p * np.log(p)).sum())), s


full_net, lora_net = nets["full"][0], nets["lora4"][0]
dW = {"full": [full_net[f"W{i}"].astype(np.float64) - base[f"W{i}"] for i in range(2)],
      "LoRA r 4": [lora_net[f"A{i}"].astype(np.float64) @ lora_net[f"B{i}"] for i in range(2)]}  # fmt: skip
spec = {}
for name, mats in dW.items():
    for i, M in enumerate(mats):
        er, s = erank(M)
        spec[name, i] = (er, s)
        rel = np.linalg.norm(M) / np.linalg.norm(base[f"W{i}"])
        print(f"{name:>8}, ΔW{i + 1} {M.shape[0]}×{M.shape[1]}: effective rank {fmt(er, 1)} (Roy–Vetterli: {fmt(erank(M, 1)[0], 1)})"
              f" · ‖ΔW‖/‖W‖ = {fmt(rel, 2)}")  # fmt: skip
full_runs = TRA["methods"]["full"]["runs"]["all"]
print(
    f"demo, full fine-tuning, 3 seeds: ΔW₁ {pts([r['erank0'] for r in full_runs])} · ΔW₂ {pts([r['erank1'] for r in full_runs])}"
)
assert 30 < spec["full", 0][0] < 70 and spec["LoRA r 4", 0][0] <= 4 + 1e-9

Not low-rank: the full fine-tuning update uses a few dozen directions (43–59 in the first layer across the 3 seeds),
while that of rank-4 LoRA uses at most 4 directions, by definition. Biderman et al. (2024) measured something
similar in language models: the full fine-tuning update has a rank 10–100 times that of the usual LoRA
configurations.

The figure below plots the singular values $\sigma_k$ of $\Delta W_1$ (log vertical axis): full fine-tuning (blue
circles) decreases gradually over a hundred-plus directions; rank-4 LoRA (orange squares) has only 4 values, then 0.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.4))
s_full, s_lora = spec["full", 0][1], spec["LoRA r 4", 0][1][:4]
ax.plot(np.arange(1, len(s_full) + 1), s_full, "o-", color=BLUE, ms=3, lw=1, label="full fine-tuning")
ax.plot(np.arange(1, 5), s_lora, "s", color=ORANGE, ms=7, label="LoRA rank 4 (the other σ are 0)")
ax.set_yscale("log")
ax.set(xlabel="direction k", ylabel="σₖ of ΔW₁", xlim=(0, 257))
ax.set_title("Singular values of the update ΔW₁ (784 × 256)", fontsize=11)
ax.grid(alpha=0.3)
ax.legend(frameon=False, fontsize=9)
fmt_axes(fig)
plt.show()
print(
    f"the first 10 directions hold {pct((s_full[:10] ** 2).sum() / (s_full**2).sum(), 1)} of the energy of the full fine-tuning ΔW₁; the first 64 hold {pct((s_full[:64] ** 2).sum() / (s_full**2).sum(), 1)}"
)

An effective rank of 45 doesn't mean that all 45 directions are needed. The first 10 directions already hold more
than half of the energy of $\Delta W_1$. The next section measures directly how many directions it takes to keep the
accuracy. One more thing: the LoRA update is not “small” at all; measured by its norm, it is even larger than the
full fine-tuning update ($\lVert\Delta W\rVert/\lVert W\rVert$ above). It is just concentrated in a few directions.

## 12 · Keeping the $k$ largest directions of $\Delta W$

The demo stores seed 0's fully fine-tuned network, so that the browser can rebuild it for any $k$:
- the $W$ of the base 0–4 network, as int8 with one float16 scale factor per column;
- the first 64 directions of $\Delta W$ for each layer, as float16: the columns of $U\,\mathrm{diag}(\sigma)$ and
  of $V$;
- the fine-tuned biases, the new output layer and the old output layer.

The network at rank $k$ uses $W_k = W + \sum_{j \le k} \sigma_j u_j v_j^\top$, that is, it keeps only the $k$ largest
directions of the update. The cell below unpacks the arrays, rebuilds the network for $k$ from 0 to 64, measures
both the new task (new output layer) and the old task (old output layer) on the whole test set, and compares with
the numbers the demo stored.

In [ ]:
def unpack(t):
    """One of the demo's arrays: float16, or int8 times a float16 factor per column (base64, little-endian)."""

    def raw(key, dt):
        return np.frombuffer(base64.b64decode(t[key]), dt).astype(np.float64)

    if "f16" in t:
        return raw("f16", "<f2").reshape(t["shape"])
    return raw("i8", np.int8).reshape(t["shape"]) * raw("scale", "<f2")


W_ship = [unpack(t) for t in SHIP["W"]]
b_ship = [unpack(t) for t in SHIP["b"]]
US, V = [unpack(t) for t in SHIP["US"]], [unpack(t) for t in SHIP["V"]]
head_new, head_old = [(unpack(SHIP[h]["W"]), unpack(SHIP[h]["b"])) for h in ("head", "oldHead")]
print("W:", [w.shape for w in W_ship], "· U·diag(σ):", [u.shape for u in US], "· V:", [v.shape for v in V])
print(
    "the first σ of ΔW₁ (column lengths of U·diag(σ)):",
    " · ".join(fmt(v, 3) for v in np.linalg.norm(US[0], axis=0)[:4]),
)
print(
    "          … stored by the demo, before float16 rounding:", " · ".join(fmt(v, 3) for v in full_runs[0]["sv0"][:4])
)


def rank_k_acc(k):
    """The accuracy of the rank-k network: the new task (5–9, new output layer) and the old task (0–4, old output
    layer)."""
    Ws = [W_ship[i] + US[i][:, :k] @ V[i][:, :k].T for i in range(2)]
    out = []
    for idx, (Wh, bh), off in ((te_new, head_new, 5), (te_old, head_old, 0)):
        h = Xt[idx].astype(np.float64)
        for i in range(2):
            h = np.maximum(h @ Ws[i] + b_ship[i], 0)
        out.append(float(((h @ Wh + bh).argmax(1) + off == yt[idx]).mean()))
    return out


shipped = TRA["shipped"]
trunc = {k: rank_k_acc(k) for k in shipped["k"]}
for j, k in enumerate(shipped["k"]):
    new, old = trunc[k]
    print(
        f"k = {k:>2}: new task {pct(new)} · old task {pct(old)}   (demo: {pct(shipped['new'][j])} / {pct(shipped['old'][j])})"
    )
    assert abs(new - shipped["new"][j]) < 3e-4 and abs(old - shipped["old"][j]) < 3e-4, "differs from the demo"
old_k = [trunc[k][1] for k in shipped["k"]]
assert old_k[-1] < old_k[0] and (np.diff(old_k) > 0).any()  # the old task drops, not at every step

Every number matches the demo. With $k = 0$ (the base encoder, but with the fine-tuned biases and output layer), the
new task drops to about half: the new output layer was trained together with $\Delta W$, so it is useless without
it. Adding 4 directions already brings it to 90.99%; 16 directions bring it to 98.15%, nearly as good as the full
network. Meanwhile the old task gradually drops, though not at every step: each added direction helps the new task,
while the old task, overall, gets a little worse.

The figure below plots two lines against $k$ (the horizontal axis on a log scale, with $k = 0$ placed at the start of
the axis): the new task (blue circles) and the old task (orange squares) of the seed-0 network, together with the
level of rank-4 LoRA trained directly (green band, 3 seeds).

In [ ]:
ks = np.array(shipped["k"])
xk = np.where(ks == 0, 0.5, ks)  # k = 0 is placed at 0.5 on the log axis
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(xk, [100 * trunc[k][0] for k in ks], "o-", color=BLUE, label="new task 5–9 (new output layer)")
ax.plot(xk, [100 * trunc[k][1] for k in ks], "s--", color=ORANGE, label="old task 0–4 (old output layer)")
lora4_new = [100 * r["new"] for r in TRA["methods"]["lora4"]["runs"]["all"]]
ax.axhspan(min(lora4_new), max(lora4_new), color=GREEN, alpha=0.35, label="LoRA rank 4, trained directly (new task)")
ax.set_xscale("log")
ax.set(xlabel="number k of directions of ΔW kept", ylabel="test accuracy (%)", ylim=(45, 101))
ax.set_title("Fully fine-tuned network (seed 0), keeping the k largest directions of ΔW", fontsize=11)
ax.grid(axis="y", alpha=0.3)
ax.legend(frameon=False, fontsize=8, loc="lower right")
fmt_axes(fig)
ax.set_xticks(xk, [str(k) for k in ks], fontsize=8)  # after fmt_axes, which replaces the log axis's ticks with 1–2–5
ax.xaxis.set_minor_locator(NullLocator())
plt.show()


def at(k):
    """The new task for the 3 seeds when k directions are kept (the demo truncates each seed's ΔW the same way)."""
    return [next(r["new"] for r in t["rows"] if r["k"] == k) for t in TRA["trunc"]["full"]]


print(
    f"the demo's 3 seeds: k = 4 → {span(at(4))} · k = 16 → {span(at(16))} · full {span([t['all'] for t in TRA['trunc']['full']])}"
)
print(f"LoRA rank 4, trained directly: {span([r['new'] for r in TRA['methods']['lora4']['runs']['all']])}")

On all 3 seeds, keeping the 16 largest directions gives 98.05–98.17% on the new task, on par with rank-4 LoRA trained
directly (98.11–98.17%) and only about half a point below the full network: that is why a small rank is usually
enough. But 4 directions cut out *after training* give only 84.12–92.61%, far below rank-4 LoRA: training directly
in rank 4 finds 4 directions that are better than the 4 largest directions of an update with no limit on its rank.
Shuttleworth et al. (2024) also found that LoRA and full fine-tuning can reach the same accuracy while learning
updates whose structures are entirely different.

## 13 · LoRA at scale: counting parameters and memory

A rank-$r$ LoRA adapter on a $d_{\text{in}} \times d_{\text{out}}$ matrix has $r(d_{\text{in}} + d_{\text{out}})$
parameters. The LoRA paper's main experiments are on GPT-3 175B: 96 blocks, $d_{\text{model}} = 12{,}288$, LoRA on
$W_q$ and $W_v$ of every block (192 matrices of size $d \times d$). The paper's formula is $|\Theta| = 2 \times \hat
L_{\text{LoRA}} \times d_{\text{model}} \times r$, where $\hat L_{\text{LoRA}}$ is the number of matrices LoRA is
added to.

In [ ]:
L_gpt3, d_gpt3, FULL_GPT3 = 96, 12_288, 175_255.8e6  # Table 4 of the LoRA paper lists 175,255.8 million parameters
for r in (1, 4, 8):
    n = sum(r * (d_gpt3 + d_gpt3) for _ in range(L_gpt3) for _ in ("W_q", "W_v"))
    assert n == 2 * (2 * L_gpt3) * d_gpt3 * r  # the paper's |Θ| formula
    print(
        f"r = {r}: {fmt(n, 0):>10} parameters · {fmt(FULL_GPT3 / n, 0)} times fewer · 16-bit file {fmt(2 * n / 1e6, 1)} MB"
    )
    assert n == {1: 4_718_592, 4: 18_874_368, 8: 37_748_736}[r]
    print(f"       ≈ {fmt(n / 1e6, 1)} million")
print(f"the whole model at 16 bits: {fmt(2 * FULL_GPT3 / 1e9, 1)} GB")

These are exactly the paper's numbers: 4.7 million and 37.7 million parameters at ranks 1 and 8 (Table 4), 18.9
million at rank 4 (Section 7.1 says “18M”), and the file saved for each new task goes from 350 GB down to about 35
MB. The number of parameters to train drops about 10,000-fold.

GPU memory, however, drops only about 3-fold: the paper measures 1.2 TB for full fine-tuning and 350 GB with LoRA. The
reason is in the last line: the frozen $W$ still has to sit in memory to run the forward and backward passes, which
is about 350 GB at 16 bits. LoRA only gets rid of the gradients and the two Adam states for $W$. **QLoRA** (Dettmers
et al., 2023) goes further: it stores the frozen $W$ in 4 bits (the NF4 type, plus a small amount for the
quantization constants), keeps LoRA in 16 bits, and can fine-tune a 65-billion-parameter model on a single 48 GB GPU.

In [ ]:
L, d, ff, V = 80, 8192, 22_016, 32_000  # LLaMA 65B's configuration: 80 blocks, d 8,192, SwiGLU MLP 22,016, vocab 32,000
n65 = (
    2 * V * d + L * (4 * d * d + 3 * d * ff) + (2 * L + 1) * d
)  # embedding + output, 4 attention matrices + 3 MLP matrices, RMSNorm
print(f"LLaMA 65B: {fmt(n65, 0)} parameters")
for name, bits in (("16 bits", 16), ("4-bit NF4 + 0.127 bits of constants", 4.127)):
    print(f"  frozen W at {name}: {fmt(n65 * bits / 8 / 1e9, 1)} GB")
assert n65 == 65_285_660_672

At 16 bits, the $W$ of LLaMA 65B alone is already about 131 GB, far more than a 48 GB GPU; 4 bits bring it down to
about 34 GB, and the rest is left for the adapter, the adapter's Adam states and the activations. The QLoRA paper
compares “more than 780 GB” for full fine-tuning at 16 bits with “less than 48 GB” for QLoRA.

## 14 · Try changing things

The cells below rerun the same functions with different parameters. You can edit the values and run them again.

### The rank of LoRA
Section 10 uses rank 4. The cell below reruns the 0–4 → 5–9 transfer with ranks 1 and 16, each with the $\eta$ that
the demo chose for seed 0, then prints the new task, the old task and the effective rank of $\Delta W_1$. Add other
ranks to `RANKS` (2 or 8) to try them; for ranks outside 1–16, choose $\eta$ yourself (for example $10^{-2}$).

In [ ]:
RANKS = (1, 16)
for r in RANKS:
    ref = next(q for q in TRA["methods"][f"lora{r}"]["runs"]["all"] if q["seed"] == 0)
    p = fit(add_lora(add_head(encoder_of(), 5, 100), r, 0), X[tr_new], y[tr_new] - 5, LORA_KEYS, ref["lr"], n_new, 0)
    new = acc(p, Xt[te_new], yt[te_new], off=5)
    old = acc(merged(p) | {"Wh": base["Wh"], "bh": base["bh"]}, Xt[te_old], yt[te_old])
    er = erank(p["A0"].astype(np.float64) @ p["B0"])[0]
    print(f"LoRA r = {r:>2} (η = {fmt(ref['lr'], 3)}): new task {pct(new)} · old task {pct(old)} · effective rank of ΔW₁ {fmt(er, 1)}"
          f"   (demo, seed 0: {pct(ref['new'])} / {pct(ref['old'])})")  # fmt: skip

Rank 1 learns the new task clearly worse than rank 4; rank 16 is nearly as good as full fine-tuning. The old task at
rank 16 varies a lot between seeds (section 10), so a single run doesn't say much. The effective rank never exceeds
$r$, and is much smaller than $r$ when the directions of $AB$ are uneven.

### SimCLR's temperature and length
Change `TAU` (section 5) or the number of steps in `simclr(steps=..., batch=...)` (section 6), then rerun those two
sections. A low temperature (0.1) penalizes the hardest negative pairs heavily; a high temperature (1) treats all
negative pairs almost the same. With more steps (1,000 steps, that is, 5 times longer), the linear probe at 100
labels gradually approaches the demo's 80.6–82.0%.

### A different random encoder
The random network of section 8 has 128 ReLU features. Try the cell below with other seeds: the probe gives almost
the same number, because a linear layer on random features measures the structure of the data more than that of the
network.

In [ ]:
for seed in (1, 2):
    te, va, lr = probe(init_encoder(seed), 5500, seed=seed, method="randprobe")
    print(
        f"random network seed {seed}: probe on {fmt(len(subset(5500, seed)), 0)} labels → {pct(te)} (demo: {pct(stored['randprobe']['55000'][seed]['test'])})"
    )

## 15 · Limits and next steps

- **Scale.** Every experiment here uses an MLP of 235 thousand parameters on MNIST. At the scale of GPT or CLIP,
  pretraining doesn't just “help when labels are scarce”: it is the only practical way to get abilities that no
  labeled set could teach.
- **Augmentation beats SimCLR here** because we know exactly which invariance we need (rotating, shifting, scaling
  digits). For text or diverse natural images, nobody can list them all, and learning from unlabeled data is the way
  to go.
- **A linear probe is only one measure.** Always put the random-network and raw-pixel baselines next to it and, if
  you can, measure full fine-tuning too.
- **Forgetting** depends on $\eta$, the number of steps and how the output layer is initialized; LoRA only reduces
  it.

One big question remains: for a given compute budget, how large a model should we pretrain, and on how much data?
The answer is scaling laws, the next lesson.

## Summary
- Self-supervision creates labels from the data itself: the next token (GPT), masked tokens (BERT, 15%), two views
  of the same image (SimCLR, NT-Xent), an image and its caption (CLIP, a symmetric loss).
- SimCLR helps when labels are scarce (100 labels: 80.6–82.0% versus 74.1–76.9% from scratch); at 55,000 labels the
  advantage is within the seed-to-seed variation. Training from scratch with SimCLR's own augmentation does even
  better (87.0–90.4%).
- A linear probe measures whether the features are linearly separable: a random network already gets 85.52–86.35%,
  raw pixels 92.79%; the autoencoder gives a low probe, but its full fine-tuning is on par with SimCLR.
- CLIP zero-shot picks the nearest caption from a list we write; the template changes the answer. On MNIST it gets
  only 48.25%, losing to softmax regression on raw pixels trained on 2 images per class.
- LoRA $h = xW + \tfrac{\alpha}{r}xAB$ ($B = 0$ at the start): rank 4 trains 6,341 instead of 234,501 parameters,
  reaches 98.1–98.2% versus 98.7–98.8% for full fine-tuning, and forgets less (old task 95.25–98.62% at ranks 2–4
  versus 80.17–94.82%).
- The full fine-tuning update is not low-rank (effective rank 43–59), but its 16 largest directions are already
  enough for 98.05–98.17%.
- GPT-3 with LoRA on $W_q, W_v$: 4,718,592 / 18,874,368 / 37,748,736 parameters at ranks 1 / 4 / 8; memory drops
  only about 3-fold because $W$ is still in the GPU.

## Review questions
1. Why doesn't BERT replace every chosen token with `[MASK]`?
2. In NT-Xent, how many candidates does a view have, and what is the loss if every $z$ is identical?
3. A new encoder gives a linear probe 2 points above a random network on MNIST. What other baselines do you need
   before saying it “learned a good representation”?
4. Why does LoRA initialize $B = 0$ rather than both $A$ and $B$ randomly? And why not initialize both to 0?
5. LoRA cuts GPT-3's number of trainable parameters about 10,000-fold. Why does GPU memory drop only about 3-fold?
6. Keeping the 4 largest directions of $\Delta W$ (after full fine-tuning) gives 84.12–92.61%, while rank-4 LoRA
   gives 98.11–98.17%. Both are rank-4 updates. Why the difference?

<details><summary>Hints and answers</summary>

1. There is no `[MASK]` at all at fine-tuning time. If it only had to predict where there is a `[MASK]`, the model
   could neglect the representations of the ordinary tokens; the 10% random tokens and the 10% left unchanged force
   it to keep a good representation at every position.
2. $2B - 1$ candidates (every other view in the batch). If every $z$ is the same, the softmax is uniform over the
   $2B - 1$ candidates, so $\ell_i = \log(2B - 1)$; with $B = 4$ that is $\log 7$.
3. A random network with the same architecture (here 85.52–86.35%) and raw pixels (92.79%): 2 points above a random
   network is still far below raw pixels. Better still, also measure full fine-tuning and training from scratch with
   the same number of labels.
4. $B = 0$ makes $AB = 0$, so at the start the network is exactly the pretrained network. If both were 0, the
   gradient of $A$ (proportional to $B$) and that of $B$ (proportional to $xA$) would both be 0, and LoRA would never
   learn.
5. The frozen $W$ still has to sit in memory (about 350 GB at 16 bits) to run the forward and backward passes. LoRA
   only gets rid of the gradients and the Adam states of $W$. QLoRA cuts further by storing $W$ in 4 bits.
6. The 4 largest directions of an update learned with no limit on its rank are not the 4 best directions for the new
   task: the rest of the update (and the output layer, trained together with it) relies on many other directions.
   LoRA trains directly in rank 4, so the output layer and the 4 directions adapt to each other.
</details>

## Further reading
- A. Radford et al. (2018), [Improving Language Understanding by Generative Pre-Training](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf) (GPT).
- J. Howard & S. Ruder (2018), [ULMFiT](https://arxiv.org/abs/1801.06146).
- J. Devlin et al. (2018), [BERT](https://arxiv.org/abs/1810.04805); Y. Liu et al. (2019),
  [RoBERTa](https://arxiv.org/abs/1907.11692).
- J. Yosinski et al. (2014), [How transferable are features in deep neural networks?](https://arxiv.org/abs/1411.1792)
- K. He, R. Girshick & P. Dollár (2018), [Rethinking ImageNet Pre-training](https://arxiv.org/abs/1811.08883);
  B. Zoph et al. (2020), [Rethinking Pre-training and Self-training](https://arxiv.org/abs/2006.06882): strong
  augmentation reduces the benefit of pretraining.
- T. Chen et al. (2020), [SimCLR](https://arxiv.org/abs/2002.05709) and [SimCLRv2](https://arxiv.org/abs/2006.10029).
- K. He et al. (2021), [Masked Autoencoders](https://arxiv.org/abs/2111.06377).
- A. Radford et al. (2021), [CLIP](https://arxiv.org/abs/2103.00020); W. Liang et al. (2022),
  [Mind the Gap](https://arxiv.org/abs/2203.02053).
- J. Hewitt & P. Liang (2019), [Designing and Interpreting Probes with Control Tasks](https://arxiv.org/abs/1909.03368);
  J. Wieting & D. Kiela (2019), [No Training Required: Exploring Random Encoders for Sentence Classification](https://arxiv.org/abs/1901.10444).
- A. Kumar et al. (2022), [Fine-Tuning can Distort Pretrained Features](https://arxiv.org/abs/2202.10054) (LP-FT).
- A. Aghajanyan, L. Zettlemoyer & S. Gupta (2020), [Intrinsic Dimensionality Explains Fine-Tuning](https://arxiv.org/abs/2012.13255).
- E. Hu et al. (2021), [LoRA](https://arxiv.org/abs/2106.09685); T. Dettmers et al. (2023),
  [QLoRA](https://arxiv.org/abs/2305.14314).
- D. Biderman et al. (2024), [LoRA Learns Less and Forgets Less](https://arxiv.org/abs/2405.09673);
  R. Shuttleworth et al. (2024), [LoRA vs Full Fine-tuning: An Illusion of Equivalence](https://arxiv.org/abs/2410.21228).
- O. Roy & M. Vetterli (2007), [The Effective Rank](https://zenodo.org/records/40328): the effective rank.
- J. Kirkpatrick et al. (2017), [Overcoming catastrophic forgetting in neural networks](https://arxiv.org/abs/1612.00796).

**Next lesson:** [Lesson 23 · Scaling laws](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/quy-luat-mo-rong/): for a
given compute budget, how large a model to pretrain, and on how much data.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")